# VGGT 3D Reconstruction Inference Pipeline (Colab Version)
This notebook allows you to run Facebook's VGGT on a folder of images using Google Colab's GPU.

In [ ]:
## 1. Setup Environment
# Note: We pin specific versions to ensure compatibility without needing to patch source code.

!pip uninstall -y pycolmap pyceres numpy

!git clone https://github.com/facebookresearch/vggt.git
%cd vggt

!pip install numpy==1.26.4
!pip install -r requirements.txt
!pip install pycolmap==3.10.0 pyceres==2.3
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install hydra-core trimesh huggingface_hub open3d plotly PILLOW

## Install specific LightGlue fork required by VGGT
!pip install git+https://github.com/jytime/LightGlue.git

print("Setup Complete!")

In [ ]:
## 2. Upload Images Zip file
import os
import shutil
import glob
from google.colab import files
from PIL import Image

print("Please upload a ZIP file containing your images (e.g., photos.zip)")
uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]

    # Extract
    !mkdir -p input_images
    !unzip -q {filename} -d input_images/ || echo "Assuming single file upload or zip extraction error..."

    # Format and Resize into VGGT scene_dir
    scene_dir = "vggt_scene"
    images_dir = os.path.join(scene_dir, "images")
    os.makedirs(images_dir, exist_ok=True)
    
    copied = 0
    for filepath in glob.glob("input_images/**/*.*", recursive=True):
        ext = os.path.splitext(filepath)[1].lower()
        if ext in ['.jpg', '.jpeg', '.png']:
            try:
                with Image.open(filepath) as img:
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    # Resize to max 512px maintaining aspect ratio
                    img.thumbnail((512, 512), Image.Resampling.LANCZOS)
                    
                    dest_path = os.path.join(images_dir, os.path.basename(filepath))
                    img.save(dest_path, quality=95)
                    copied += 1
            except Exception as e:
                print(f"Error processing {filepath}: {e}")
            
    print(f"Done formatting and resizing {copied} images at {scene_dir}. Prepared for inference.")
else:
    print("No file uploaded.")

In [ ]:
## 3. Run VGGT Inference
!python demo_colmap.py --scene_dir vggt_scene

In [ ]:
## 4. Visualize Result (Point Cloud)
import open3d as o3d
import plotly.graph_objects as go
import numpy as np
import os

ply_path = "vggt_scene/sparse/points.ply"

if os.path.exists(ply_path):
    print("Loading point cloud...")
    pcd = o3d.io.read_point_cloud(ply_path)
    
    # Downsample for faster notebook rendering if too large
    if len(pcd.points) > 100000:
        print(f"Original points: {len(pcd.points)}. Downsampling for visualization...")
        pcd = pcd.voxel_down_sample(voxel_size=0.02)
        
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None

    # Create Plotly character
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode='markers',
        marker=dict(
            size=1.5,
            color=colors if colors is not None else points[:, 2], # Use Z-coord if no color
            opacity=0.8
        )
    )])

    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False)
        ),
        margin=dict(l=0, r=0, b=0, t=0)
    )
    fig.show()
else:
    print(f"Error: Points file not found at {ply_path}. Please check inference output.")

In [ ]:
## 5. Download Results
import shutil
from google.colab import files
# zip the sparse folder which contains points.ply and cameras/images/points3D bins
shutil.make_archive('vggt_reconstruction', 'zip', 'vggt_scene/sparse')
files.download('vggt_reconstruction.zip')
print("Downloading results... You can now run the scale evaluator locally on the downloaded points.ply file.")